In [5]:
'''
SEQUENTIAL AGENT:

The SequentialAgent is a workflow agent. It's not powered by an LLM itself; instead, its only job is to execute a list of other agents in a strict, predefined order.

The real magic ✨ is how it passes information. The ADK uses a shared state dictionary that each agent in the sequence can read from and write to.

Our New Workflow:

Foodie Agent: Finds the restaurant and saves the name to state['destination'].
Transportation Agent: Automatically reads state['destination'] and uses it to find directions.
This means we no longer need custom Python code to extract text or build new queries! The ADK handles the plumbing for us.
'''


"\nSEQUENTIAL AGENT:\n\nThe SequentialAgent is a workflow agent. It's not powered by an LLM itself; instead, its only job is to execute a list of other agents in a strict, predefined order.\n\nThe real magic ✨ is how it passes information. The ADK uses a shared state dictionary that each agent in the sequence can read from and write to.\n\nOur New Workflow:\n\nFoodie Agent: Finds the restaurant and saves the name to state['destination'].\nTransportation Agent: Automatically reads state['destination'] and uses it to find directions.\nThis means we no longer need custom Python code to extract text or build new queries! The ADK handles the plumbing for us.\n"

In [6]:
import os
import sys
import  json
import asyncio
import random
import string
from uuid import uuid4
from typing import List,Any
from IPython.display import HTML, Markdown, display

#----------ADK , Agent and Evaluation components Tools Contextimports here------------------

from google.adk.agents import Agent, SequentialAgent
from google.adk.events import Event
from google.adk.runners import Runner
import google.adk as adk
from google.adk.tools import google_search, ToolContext
from google.adk.sessions import InMemorySessionService, Session
from google.genai import types
from google.genai.types import Content , Part

from dotenv import load_dotenv

print(" All libraries are imported!")

 All libraries are imported!


In [7]:
load_dotenv()

True

In [8]:
#Runner to Help run the agent: This is a HELPER function
async def run_agent_query(agent:Agent,query:str,session : Session,user_id: str,is_router: bool = False):
    """Initializes a runner and executes a query for a given agent and session."""
    print(f"\n Running query for agent: '{agent.name}' in session: '{session.id}'...")
    runner = Runner(
        agent = agent,
        session_service = session_service,
        app_name = agent.name)
    final_response = ""
    try:
        async for event in runner.run_async(user_id = user_id ,session_id = session.id,new_message = Content(parts=[Part(text = query)],role ="user")):
            if not is_router:
                # Let's see what the agent is thinking! through events 
                print(f"EVENT:{event}")
                if event.is_final_response():
                    final_response = event.content.parts[0].text
    except Exception as e:
        final_response = f"An error occurred: {e}"

    if not is_router:
        print("\n" + "-"*50)
        print("✅ Final Response:")
        display(Markdown(final_response))
        print("-"*50 + "\n")
    return final_response

In [9]:
# --- Initializing Session Service ---
session_service = InMemorySessionService()
my_user_id = "adk_user_001"

In [10]:
db_agent = Agent(
    name = "db_agent",
    model = "gemini-3.5-flash",
    instruction = "You are a database agent. When asked for data, return this mock JSON object: {'status': 'success', 'data': [{'name': 'The Grand Hotel', 'rating': 5, 'reviews': 450}, {'name': 'Seaside Inn', 'rating': 4, 'reviews': 620}]}"
)

In [12]:
#----------------Agent Definitions for our Specialist Team (Refactored for Sequential Workflow)-------------

#agent1 :foodie_agent to save its output to the shared state. `output_key` and the more specific instruction
foodie_agent = Agent(
    name = "foodie_agent",
    model = "gemini-3.5-flash",
    tools = [google_search],
    instruction = """
    You are an expert food critic. Your goal is to find the best restaurant based on a user's request.
    When you recommend a place, you must output *only* the name of the establishment and nothing else.
    For example, if the best sushi is at 'Jin Sho', you should output only: Jin Sho
    """,
    output_key = "destination"# ADK will save the agent's final response to state['destination']
)

#agent2 : transportation_agent to read from the shared state. The `{destination}` placeholder is automatically filled by the ADK from the state.
transportation_agent = Agent(
    name = "transportation_agent",
    model = "gemini-3.5-flash",
    tools = [google_search],
    instruction = """
    "You are a navigation assistant. Given a destination, provide clear directions.
    The user wants to go to: {destination}.
    Analyze the user's full original query to find their starting point.
    Then, provide clear directions from that starting point to {destination}.
    """,
)
#agent3: day_trip_agent
day_trip_agent = Agent(
    name="day_trip_agent",
    model="gemini-2.5-flash",
    description="Agent specialized in generating spontaneous full-day itineraries based on mood, interests, and budget.",
    instruction="""
    You are the "Spontaneous Day Trip" Generator 🚗 - a specialized AI assistant that creates engaging full-day itineraries.

    Your Mission:
    Transform a simple mood or interest into a complete day-trip adventure with real-time details, while respecting a budget.

    Guidelines:
    1. **Budget-Aware**: Pay close attention to budget hints like 'cheap', 'affordable', or 'splurge'. Use Google Search to find activities (free museums, parks, paid attractions) that match the user's budget.
    2. **Full-Day Structure**: Create morning, afternoon, and evening activities.
    3. **Real-Time Focus**: Search for current operating hours and special events.
    4. **Mood Matching**: Align suggestions with the requested mood (adventurous, relaxing, artsy, etc.).

    RETURN itinerary in MARKDOWN FORMAT with clear time blocks and specific venue names.
    """,
    tools=[google_search]
)

#Defining the SequentialAgent to manage the workflow.This agent will run foodie_agent, then transportation_agent, in that exact order.
find_and_navigate_agent = SequentialAgent(
    name = "find_and_navigate_agent",
    sub_agents = [foodie_agent,transportation_agent],
    description = "A workflow that first finds a location and then provides directions to it."
)

weekend_guide_agent = Agent(
    name = "weekend_guide_agent",
    model = "gemini-3.5-flash",
    tools = [google_search],
    instruction="You are a local events guide. Your task is to find interesting events, concerts, festivals, and activities happening on a specific weekend."
)

#updating router agent 
router_agent = Agent(
    name = "router_agent",
    model = "gemini-3.5-flash",
    instruction="""
    You are a request router. Your job is to analyze a user's query and decide which of the following agents or workflows is best suited to handle it.
    Do not answer the query yourself, only return the name of the most appropriate choice.

    Available Options:
    - 'foodie_agent': For queries *only* about food, restaurants, or eating.
    - 'weekend_guide_agent': For queries about events, concerts, or activities happening on a specific timeframe like a weekend.
    - 'day_trip_agent': A general planner for any other day trip requests.
    - 'find_and_navigate_agent': Use this for complex queries that ask to *first find a place* and *then get directions* to it.

    Only return the single, most appropriate option's name and nothing else.
    """
)

#dictionary of all our executable agents for easy lookup.
worker_agents = {
    "day_trip_agent": day_trip_agent,
    "foodie_agent": foodie_agent,
    "weekend_guide_agent": weekend_guide_agent,
    "find_and_navigate_agent": find_and_navigate_agent,
}
print("Agent team assembled with a SequentialAgent workflow!")

Agent team assembled with a SequentialAgent workflow!


C:\Users\Lenovo\AppData\Local\Temp\ipykernel_15196\2774425918.py:51: DeprecationWarning: SequentialAgent is deprecated in favor of Workflow and will be removed in a future version. Workflow cannot yet be used as an LlmAgent sub-agent.
  find_and_navigate_agent = SequentialAgent(


In [13]:
# ------------------Testing the Streamlined Workflow-----------------
async def run_sequential_app():
    queries = [
        "I want to eat the best sushi in Palo Alto.", # Should go to foodie_agent
        "Are there any cool outdoor concerts this weekend?", # Should go to weekend_guide_agent
        "Find me the best sushi in Palo Alto and then tell me how to get there from the Caltrain station." # Should trigger the SequentialAgent
    ]

    for query in queries:
        print(f"\n{'='*60}\nProcessing New Query: '{query}'\n{'='*60}")

        # 1. Ask the Router Agent to choose the right agent or workflow
        router_session = await session_service.create_session(app_name=router_agent.name, user_id=my_user_id)
        print("Asking the router agent to make a decision...")
        chosen_route = await run_agent_query(router_agent, query, router_session, my_user_id, is_router=True)
        chosen_route = chosen_route.strip().replace("'", "")
        print(f"Router has selected route: '{chosen_route}'")

        # 2. Execute the chosen route
        # This logic is now much simpler! The SequentialAgent is treated just like any other worker.
        if chosen_route in worker_agents:
            worker_agent = worker_agents[chosen_route]
            print(f"--- Handing off to {worker_agent.name} ---")
            worker_session = await session_service.create_session(app_name=worker_agent.name, user_id=my_user_id)
            await run_agent_query(worker_agent, query, worker_session, my_user_id)
            print(f"--- {worker_agent.name} Complete ---")
        else:
            print(f"Error: Router chose an unknown route: '{chosen_route}'")

await run_sequential_app()


Processing New Query: 'I want to eat the best sushi in Palo Alto.'
Asking the router agent to make a decision...

 Running query for agent: 'router_agent' in session: '57b22184-2bf7-48f6-a025-26db7718dce7'...


Direct use of automatic function calling (AFC) in AsyncModels.generate_content is not recommended. Instead, we recommend to use AFC in AsyncChat.send_message. Similarly, direct use of AFC in AsyncModels.generate_content_stream is not recommended. Instead, we recommend to use AFC in AsyncChat.send_message_stream.


Router has selected route: ''


NameError: name 'worker_agents' is not defined